# DMV Mobility Flow Map
Pre-COVID (2018-2019) vs COVID Era (2020-2021)

Outputs two CSVs for manual upload to https://kepler.gl/demo

**In Kepler:**
- Add as a new layer → Line layer
- Set origin: `origin_lat`, `origin_lng`
- Set destination: `dest_lat`, `dest_lng`
- Color by: `delta_pm25` (diverging scale — red = worse air, blue = cleaner)
- Stroke width by: `avg_visitor_count`

Requirements: `pip install geopandas pygris`

In [1]:
import pandas as pd
import geopandas as gpd
import pygris
from pathlib import Path

FLOW_FILE = Path('../data/processed/flow_map.csv')
GHAP_FILE = Path('../data/processed/ghap_pm25_monthly.csv')
OUT_DIR   = Path('../data/processed')
OUT_DIR.mkdir(exist_ok=True)

DMV_COUNTIES = [
    '11001',
    '24031', '24033',
    '51013', '51059',
    '51510', '51600', '51610',
]

MIN_VISITORS = 20

## 1. Load CBG Centroids

In [2]:
print('Loading CBG boundaries...')
parts = [pygris.block_groups(state=s, year=2019, cache=True) for s in ['DC', 'MD', 'VA']]
cbgs  = pd.concat(parts, ignore_index=True)
cbgs  = gpd.GeoDataFrame(cbgs, crs=cbgs.crs)
cbgs  = cbgs[cbgs['GEOID'].str[:5].isin(DMV_COUNTIES)][['GEOID', 'geometry']].copy()

# Project to meters for accurate centroid then back to lat/lng
cbgs_proj   = cbgs.to_crs('EPSG:32618')
cbgs['lng'] = cbgs_proj.geometry.centroid.to_crs('EPSG:4326').x
cbgs['lat'] = cbgs_proj.geometry.centroid.to_crs('EPSG:4326').y
centroids   = cbgs[['GEOID', 'lat', 'lng']].copy()

print(f'  {len(centroids):,} CBG centroids')

Loading CBG boundaries...
Using FIPS code '11' for input 'DC'
Using FIPS code '24' for input 'MD'
Using FIPS code '51' for input 'VA'
  2,548 CBG centroids


## 2. Load GHAP PM2.5

In [3]:
ghap = pd.read_csv(GHAP_FILE, dtype={'GEOID': str})
ghap['GEOID'] = ghap['GEOID'].str.zfill(12)

ghap_avg = (
    ghap.groupby('GEOID')['pm25_avg']
    .mean()
    .reset_index()
    .rename(columns={'pm25_avg': 'pm25_mean'})
)

print(f'  {len(ghap_avg):,} CBGs  |  range {ghap_avg["pm25_mean"].min():.2f} - {ghap_avg["pm25_mean"].max():.2f} ug/m3')

  2,548 CBGs  |  range 8.27 - 18.25 ug/m3


## 3. Load Flow Data

In [4]:
flows = pd.read_csv(FLOW_FILE, dtype={'home_cbg': str, 'poi_cbg': str}, low_memory=False)
flows['home_cbg'] = flows['home_cbg'].str.zfill(12)
flows['poi_cbg']  = flows['poi_cbg'].str.zfill(12)

print(f'Total rows : {len(flows):,}')
print(f'Periods    : {flows["period"].unique().tolist()}')

pre_covid = flows[flows['period'] == 'Pre-COVID (2018-2019)'].copy()
covid     = flows[flows['period'] == 'COVID Era (2020-2021)'].copy()

print(f'Pre-COVID : {len(pre_covid):,} pairs')
print(f'COVID Era : {len(covid):,} pairs')

Total rows : 6,200,529
Periods    : ['All Years', 'COVID Era (2020-2021)', 'Pre-COVID (2018-2019)']
Pre-COVID : 2,173,532 pairs
COVID Era : 1,511,254 pairs


## 4. Join Coordinates and Compute Delta PM2.5 Per Flow

In [5]:
def prepare_flows(df, centroids, ghap_avg, min_visitors=20):
    # Coordinates
    df = df.merge(centroids.rename(columns={'GEOID': 'home_cbg', 'lat': 'origin_lat', 'lng': 'origin_lng'}), on='home_cbg', how='left')
    df = df.merge(centroids.rename(columns={'GEOID': 'poi_cbg',  'lat': 'dest_lat',   'lng': 'dest_lng'}),   on='poi_cbg',  how='left')
    # PM2.5
    df = df.merge(ghap_avg.rename(columns={'GEOID': 'home_cbg', 'pm25_mean': 'pm25_origin'}), on='home_cbg', how='left')
    df = df.merge(ghap_avg.rename(columns={'GEOID': 'poi_cbg',  'pm25_mean': 'pm25_dest'}),   on='poi_cbg',  how='left')
    # Delta
    df['delta_pm25'] = df['pm25_dest'] - df['pm25_origin']
    # Clean
    df = df.dropna(subset=['origin_lat', 'origin_lng', 'dest_lat', 'dest_lng', 'delta_pm25'])
    df = df[df['home_cbg'] != df['poi_cbg']]
    df = df[df['avg_visitor_count'] >= min_visitors]
    return df.copy()


pre_covid = prepare_flows(pre_covid, centroids, ghap_avg, MIN_VISITORS)
covid     = prepare_flows(covid,     centroids, ghap_avg, MIN_VISITORS)

print(f'Pre-COVID flows : {len(pre_covid):,}')
print(f'COVID flows     : {len(covid):,}')

Pre-COVID flows : 44,202
COVID flows     : 21,903


## 5. Export CSVs for Kepler.gl

In [6]:
cols = [
    'home_cbg', 'poi_cbg',
    'origin_lat', 'origin_lng',
    'dest_lat', 'dest_lng',
    'avg_visitor_count', 'avg_median_dwell',
    'delta_pm25', 'pm25_origin', 'pm25_dest'
]

pre_covid[cols].to_csv(OUT_DIR / 'kepler_pre_covid.csv', index=False)
covid[cols].to_csv(OUT_DIR / 'kepler_covid.csv', index=False)

print(f'kepler_pre_covid.csv  — {len(pre_covid):,} rows')
print(f'kepler_covid.csv      — {len(covid):,} rows')

kepler_pre_covid.csv  — 44,202 rows
kepler_covid.csv      — 21,903 rows


## 6. Summary

In [7]:
for label, df in [('Pre-COVID (2018-2019)', pre_covid), ('COVID Era (2020-2021)', covid)]:
    pct_worse = (df['delta_pm25'] > 0).mean() * 100
    print(f'{label}')
    print(f'  Flow lines       : {len(df):,}')
    print(f'  Total visitors   : {df["avg_visitor_count"].sum():,.0f}')
    print(f'  Mean delta PM2.5 : {df["delta_pm25"].mean():.3f} ug/m3')
    print(f'  Pct worse air    : {pct_worse:.1f}%')
    print()

pct_change = (covid['avg_visitor_count'].sum() - pre_covid['avg_visitor_count'].sum()) / pre_covid['avg_visitor_count'].sum() * 100
print(f'Mobility change COVID vs Pre-COVID: {pct_change:+.1f}%')

Pre-COVID (2018-2019)
  Flow lines       : 44,202
  Total visitors   : 2,259,914
  Mean delta PM2.5 : 0.547 ug/m3
  Pct worse air    : 64.7%

COVID Era (2020-2021)
  Flow lines       : 21,903
  Total visitors   : 1,065,221
  Mean delta PM2.5 : 0.366 ug/m3
  Pct worse air    : 63.8%

Mobility change COVID vs Pre-COVID: -52.9%
